In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import models
from tensorflow.keras.models import Sequential
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.layers import TextVectorization,Embedding,LSTM,Dense,Dropout,Bidirectional
from tensorflow.keras.optimizers import Adam
import re

In [47]:
def clean_text(text):
    text = str(text)
    # Remove URLs, mentions, special characters (keep letters and spaces)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s\u0600-\u06FF]', '', text)  # Keep English/Arabic letters
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [48]:
data['cleaned_text'] =data['Text'].apply(clean_text)

In [50]:
x=data['cleaned_text'].values
y=data['Label'].values

In [52]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, shuffle=True, random_state=42,stratify=y)

In [62]:
unique_labels, counts = np.unique(y_test, return_counts=True)

In [63]:
unique_labels, counts

(array([0, 1]), array([23488,  1533]))

In [68]:
VOCAB_SIZE =10000
MAX_LEN=100

In [71]:
tokenizer=Tokenizer(num_words=VOCAB_SIZE,oov_token='<OOV>')
tokenizer.fit_on_texts(x_train)

# Sequnces
train_sequence = tokenizer.texts_to_sequences(x_train)
test_sequence=tokenizer.texts_to_sequences(x_test)

# Padding and truncation
train_padded=pad_sequences(train_sequence,maxlen= MAX_LEN,padding='post',truncating='post')
test_padded=pad_sequences(test_sequence,maxlen= MAX_LEN,padding='post',truncating='post')

In [74]:
with open('/content/drive/MyDrive/BesafeLArgedataTokenize.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [80]:
EMBEDDING_DIM =128
LSTM_UNITS =32
# Model parameters

model = models.Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS, dropout=0.3, return_sequences=False)),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
model.build(input_shape=(None, MAX_LEN))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [81]:
callbacks=[
    tf.keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(filepath='/content/drive/MyDrive/BEsafe_models/BesageLargedataModel.h5',save_best_only=True)
]

In [ ]:
history =model.fit(train_padded,y_train,epochs=60,shuffle=True,validation_split=0.3,callbacks=callbacks,batch_size=32)

In [83]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.show()

NameError: name 'history' is not defined

<Figure size 1000x600 with 0 Axes>